# Denoising Diffusion (DDPM) basics

Diffusion models learn to generate data by learning to reverse a gradual noising process.
We implement the core DDPM (Ho et al. 2020) algorithm from scratch on 2D points instead of
images - same math, much faster to train, and easy to check numerically whether it actually
learned the target distribution (no eyeballing pixels required).

Target data: points scattered in a ring of radius 2. Starting from pure Gaussian noise, the
trained model should learn to "denoise" its way back to that ring.

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

T = 200  # number of diffusion steps
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)
alpha_bars_prev = torch.cat([torch.tensor([1.0], device=device), alpha_bars[:-1]])
posterior_var = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)


def sample_ring(n, R=2.0, noise=0.1):
    theta = torch.rand(n, device=device) * 2 * np.pi
    r = R + torch.randn(n, device=device) * noise
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

## Forward process

The forward process adds noise in closed form - no need to iterate step by step during
training:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

The network's job is to predict the noise $\epsilon$ that was added, given the noisy point
$x_t$ and the timestep $t$.

In [ ]:
class EpsModel(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        t_norm = (t.float() / T).unsqueeze(1)
        return self.net(torch.cat([x, t_norm], dim=1))


model = EpsModel().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
batch = 256

for step in range(2001):
    x0 = sample_ring(batch)
    t = torch.randint(0, T, (batch,), device=device)
    eps = torch.randn_like(x0)
    ab = alpha_bars[t].unsqueeze(1)
    x_t = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps

    eps_pred = model(x_t, t)
    loss = torch.mean((eps_pred - eps) ** 2)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 400 == 0:
        print(f"step {step:5d}  loss {loss.item():.4f}")

## Reverse process (sampling)

Start from pure noise $x_T \sim \mathcal{N}(0, I)$ and iteratively denoise using the standard
DDPM update:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,
\epsilon_\theta(x_t, t)\right) + \sigma_t z, \qquad z \sim \mathcal{N}(0, I) \text{ if } t > 0
\text{ else } 0$$

using the true posterior variance $\sigma_t^2 = \tilde\beta_t$ from the DDPM paper.

In [ ]:
with torch.no_grad():
    n_samples = 256
    x = torch.randn(n_samples, 2, device=device)
    r0 = x.norm(dim=1)
    print(f"initial (pure noise) radius: mean {r0.mean():.3f}  std {r0.std():.3f}")

    for t_idx in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long, device=device)
        eps_pred = model(x, t_batch)
        alpha_t, alpha_bar_t, beta_t = alphas[t_idx], alpha_bars[t_idx], betas[t_idx]
        mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * eps_pred)
        if t_idx > 0:
            x = mean + torch.sqrt(posterior_var[t_idx]) * torch.randn_like(x)
        else:
            x = mean

    r_final = x.norm(dim=1)
    print(f"final (sampled) radius:      mean {r_final.mean():.3f}  std {r_final.std():.3f}")
    print("target ring: radius 2.0, noise std 0.1")

In [ ]:
import matplotlib.pyplot as plt

real = sample_ring(256).cpu()
generated = x.cpu()

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
axes[0].scatter(real[:, 0], real[:, 1], s=8, alpha=0.6)
axes[0].set_title("real data")
axes[1].scatter(generated[:, 0], generated[:, 1], s=8, alpha=0.6, color="tab:orange")
axes[1].set_title("generated (from pure noise)")
for ax in axes:
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
plt.show()

## Next steps

- Swap `sample_ring` for a different toy distribution (two moons, a mixture of Gaussians, a
  spiral) and retrain.
- Move from 2D points to small images: replace `EpsModel` with a small U-Net that takes an
  image and timestep and predicts pixel-wise noise - see `unet.ipynb` for the encoder/decoder
  building block this needs, and condition it on `t` the same way `EpsModel` does here.
- Try fewer reverse steps at sampling time (e.g. skip every other `t`) and see how much sample
  quality degrades - this tradeoff is exactly what faster samplers (DDIM, etc.) optimize.